In [1]:
import sqlite3

In [ ]:
#Se o sqlite não estiver instalado, descomente e execute:

#pip install sqlite3

In [2]:
conexao = sqlite3.connect('exercicio_sql.db')
cursor = conexao.cursor()

cursor.executescript('''
-- ==========================================
-- 1. CRIAÇÃO DAS TABELAS
-- ==========================================

CREATE TABLE departamentos (
    id INT PRIMARY KEY,
    nome VARCHAR(50) NOT NULL
);

CREATE TABLE funcionarios (
    id INT PRIMARY KEY,
    nome VARCHAR(100) NOT NULL,
    salario DECIMAL(10,2) NOT NULL,
    departamento_id INT,
    supervisor_id INT,
    data_contratacao DATE,
    FOREIGN KEY (departamento_id) REFERENCES departamentos(id),
    FOREIGN KEY (supervisor_id) REFERENCES funcionarios(id)
);

CREATE TABLE bonus_funcionarios (
    funcionario_id INT,
    valor_bonus DECIMAL(10,2),
    data_bonus DATE,
    FOREIGN KEY (funcionario_id) REFERENCES funcionarios(id)
);

CREATE TABLE categorias (
    id INT PRIMARY KEY,
    nome VARCHAR(50) NOT NULL
);

CREATE TABLE produtos (
    id INT PRIMARY KEY,
    nome VARCHAR(100) NOT NULL,
    preco DECIMAL(10,2) NOT NULL,
    estoque INT NOT NULL,
    categoria_id INT,
    FOREIGN KEY (categoria_id) REFERENCES categorias(id)
);

CREATE TABLE clientes (
    id INT PRIMARY KEY,
    nome VARCHAR(100) NOT NULL,
    email VARCHAR(100),
    saldo DECIMAL(10,2) DEFAULT 0.00
);

CREATE TABLE pedidos (
    id INT PRIMARY KEY,
    cliente_id INT,
    data_pedido DATE NOT NULL,
    metodo_envio VARCHAR(50),
    FOREIGN KEY (cliente_id) REFERENCES clientes(id)
);

CREATE TABLE itens_pedido (
    pedido_id INT,
    produto_id INT,
    quantidade INT NOT NULL,
    preco_unitario DECIMAL(10,2) NOT NULL,
    PRIMARY KEY (pedido_id, produto_id),
    FOREIGN KEY (pedido_id) REFERENCES pedidos(id),
    FOREIGN KEY (produto_id) REFERENCES produtos(id)
);

-- Tabelas adicionais para os exercícios difíceis de JOIN
CREATE TABLE vendas_2024 (
    produto_id INT PRIMARY KEY,
    receita DECIMAL(12,2)
);

CREATE TABLE vendas_2025 (
    produto_id INT PRIMARY KEY,
    receita DECIMAL(12,2)
);

CREATE TABLE cores ( cor VARCHAR(30) );
CREATE TABLE tamanhos ( tamanho VARCHAR(5) );

CREATE TABLE logins_usuarios (
    usuario_id INT,
    data_login DATE
);

-- ==========================================
-- 2. POPULANDO OS DADOS (INSERT INTO)
-- ==========================================

-- Departamentos
INSERT INTO departamentos VALUES (1, 'TI');
INSERT INTO departamentos VALUES (2, 'Vendas');
INSERT INTO departamentos VALUES (3, 'RH');
INSERT INTO departamentos VALUES (4, 'Marketing');
INSERT INTO departamentos VALUES (5, 'Diretoria sem Funcionarios'); -- Proposital para JOINs

-- Funcionários (Hierarquia e salários)
INSERT INTO funcionarios VALUES (1, 'Diretor Carlos', 15000.00, 1, NULL, '2020-01-15');
INSERT INTO funcionarios VALUES (2, 'Gerente Ana', 9000.00, 1, 1, '2022-03-10');
INSERT INTO funcionarios VALUES (3, 'Dev Bruno', 6000.00, 1, 2, '2024-05-20');
INSERT INTO funcionarios VALUES (4, 'Dev Cláudia', 6500.00, 1, 2, '2026-01-10'); -- Ano Atual
INSERT INTO funcionarios VALUES (5, 'Gerente Roberta', 8500.00, 2, 1, '2021-08-11');
INSERT INTO funcionarios VALUES (6, 'Vendedor Marcos', 3000.00, 2, 5, '2023-02-15');
INSERT INTO funcionarios VALUES (7, 'Vendedor Letícia', 3200.00, 2, 5, '2026-02-01'); -- Ano Atual
INSERT INTO funcionarios VALUES (8, 'Analista Igor', 4000.00, 3, 1, '2025-07-01');

-- Bônus
INSERT INTO bonus_funcionarios VALUES (3, 500.00, '2026-05-01');
INSERT INTO bonus_funcionarios VALUES (6, 1200.00, '2026-05-01');
INSERT INTO bonus_funcionarios VALUES (7, 1500.00, '2026-05-01');

-- Categorias
INSERT INTO categorias VALUES (1, 'Eletrônicos');
INSERT INTO categorias VALUES (2, 'Livros');
INSERT INTO categorias VALUES (3, 'Roupas');
INSERT INTO categorias VALUES (4, 'Casa e Decoração');
INSERT INTO categorias VALUES (5, 'Informatica');

-- Produtos
INSERT INTO produtos VALUES (101, 'Smartphone X', 2500.00, 50, 1);
INSERT INTO produtos VALUES (102, 'Notebook Nitro', 4500.00, 15, 1);
INSERT INTO produtos VALUES (103, 'Livro SQL Avançado', 120.00, 100, 2);
INSERT INTO produtos VALUES (104, 'Livro Dom Casmurro', 45.00, 0, 2); -- Estoque zerado de propósito
INSERT INTO produtos VALUES (105, 'Camiseta Algodão', 79.90, 200, 3);
INSERT INTO produtos VALUES (106, 'Caneca Geek', 35.00, 40, NULL); -- Sem categoria de propósito

-- Clientes
INSERT INTO clientes VALUES (1, 'João Silva', 'joao@email.com', 150.00);
INSERT INTO clientes VALUES (2, 'Maria Santos', 'maria@email.com', 6000.00); -- VIP em potencial
INSERT INTO clientes VALUES (3, 'Pedro Souza', 'pedro@email.com', 20.00);
INSERT INTO clientes VALUES (4, 'Ana Oliveira', 'ana.oli@email.com', 1200.00);
INSERT INTO clientes VALUES (42, 'Guia do Mochileiro', '42@email.com', 4200.00); -- Citado no Ex 20
INSERT INTO clientes VALUES (5, 'Carlos Lima', 'carlos@email.com', 0.00); -- Cliente sem pedido

-- Pedidos
INSERT INTO pedidos VALUES (1001, 1, '2026-05-10', 'Sedex');
INSERT INTO pedidos VALUES (1002, 1, '2026-06-01', 'PAC'); -- Data do Ex 17
INSERT INTO pedidos VALUES (1003, 2, '2026-06-01', 'Sedex'); -- Data do Ex 17
INSERT INTO pedidos VALUES (1004, 3, '2026-04-15', NULL); -- Sem envio ainda
INSERT INTO pedidos VALUES (1005, 2, '2026-05-20', 'Retirada');
INSERT INTO pedidos VALUES (1006, 2, '2026-06-05', 'Sedex');
INSERT INTO pedidos VALUES (1007, 4, '2026-06-09', 'PAC');

-- Itens dos Pedidos
INSERT INTO itens_pedido VALUES (1001, 101, 1, 2500.00);
INSERT INTO itens_pedido VALUES (1001, 103, 2, 120.00);
INSERT INTO itens_pedido VALUES (1002, 105, 3, 79.90);
INSERT INTO itens_pedido VALUES (1003, 102, 1, 4500.00);
INSERT INTO itens_pedido VALUES (1004, 103, 1, 120.00);
INSERT INTO itens_pedido VALUES (1005, 101, 1, 2500.00);
INSERT INTO itens_pedido VALUES (1006, 105, 1, 79.90);
INSERT INTO itens_pedido VALUES (1007, 104, 1, 45.00);

-- Dados para os exercícios difíceis de comparação anual (Ex 11)
INSERT INTO vendas_2024 VALUES (101, 50000.00);
INSERT INTO vendas_2024 VALUES (102, 90000.00);
INSERT INTO vendas_2024 VALUES (103, 1200.00);

INSERT INTO vendas_2025 VALUES (101, 75000.00); -- Aumentou
INSERT INTO vendas_2025 VALUES (102, 45000.00); -- Diminuiu
INSERT INTO vendas_2025 VALUES (105, 15000.00); -- Novo produto em 2025

-- Cross Join (Ex 12)
INSERT INTO cores VALUES ('Vermelho'), ('Azul'), ('Verde');
INSERT INTO tamanhos VALUES ('P'), ('M'), ('G');

-- Logins (Para exercícios de Gaps & Islands e Subqueries - Ex 13, 73)
INSERT INTO logins_usuarios VALUES (1, '2026-06-01');
INSERT INTO logins_usuarios VALUES (1, '2026-06-02');
INSERT INTO logins_usuarios VALUES (1, '2026-06-03'); -- Sequência de 3 dias
INSERT INTO logins_usuarios VALUES (2, '2026-06-01');
INSERT INTO logins_usuarios VALUES (2, '2026-06-05');
INSERT INTO logins_usuarios VALUES (5, '2026-06-10'); -- Logou mas não comprou
                     ''')